# Khám Phá & Đánh Giá Semantic Vector Search Trên Legal Knowledge Graph

Notebook này minh họa và đánh giá khả năng **Vector Search (Semantic Search)** tích hợp trực tiếp vào **Neo4j** bằng mô hình embedding `BAAI/bge-m3` (1024 chiều):

1. **Kiểm tra trạng thái Vector Index** (`article_embeddings_idx`) và dữ liệu embedding của 912 `Article` nodes.
2. **Thực thi Semantic Search** cho các tình huống pháp lý bằng ngôn ngữ tự nhiên.
3. **Graph Expansion (Hybrid Retrieval)**: Từ các Điều luật tìm thấy qua Vector Search, duyệt tiếp các quan hệ đồ thị (`CONTAINS` $\rightarrow$ Khoản, Điểm; `REFERENCES` $\rightarrow$ Điều luật liên quan) để thu thập ngữ cảnh toàn diện phục vụ RAG Reasoning.

In [ ]:
import os
import sys
import time
import pathlib
import pandas as pd
from dotenv import load_dotenv

# Đảm bảo import được module từ thư mục gốc backend
BACKEND_DIR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.graph.embedder import NodeEmbedder
from src.graph.neo4j_client import Neo4jClient

client = Neo4jClient()
embedder = NodeEmbedder()
print(f"✓ Đã nạp cấu hình: Model = {embedder.model_name}, Dimension = {embedder.dimension}")

In [ ]:
# 1. Thống kê tổng số Article nodes và số lượng đã có embedding vector
stats_query = """
MATCH (a:Article)
RETURN count(a) AS total_articles,
       count(a.embedding) AS embedded_articles,
       count(a) - count(a.embedding) AS missing_embeddings
"""
stats = client.execute_read(stats_query)[0]
print(f"Tổng số Article nodes: {stats['total_articles']:,}")
print(f"Số Article đã có vector: {stats['embedded_articles']:,}")
print(f"Số Article chưa có vector: {stats['missing_embeddings']:,}")

# 2. Kiểm tra trạng thái Vector Index
idx_query = """
SHOW VECTOR INDEXES YIELD name, state, populationPercent, options
WHERE name = 'article_embeddings_idx'
RETURN name, state, populationPercent, options
"""
idx_info = client.execute_read(idx_query)
if idx_info:
    cfg = idx_info[0].get('options', {}).get('indexConfig', {})
    print(f"Vector Index: {idx_info[0]['name']} | Trạng thái: {idx_info[0]['state']} | Populated: {idx_info[0]['populationPercent']}%")
    print(f"Cấu hình: Dimension = {cfg.get('vector.dimensions')} | Similarity = {cfg.get('vector.similarity_function')}")
else:
    print("Chưa tìm thấy Vector Index 'article_embeddings_idx'")

In [ ]:
def vector_search(query_text: str, top_k: int = 5) -> pd.DataFrame:
    """
    Hàm tìm kiếm Top-K Article nodes liên quan nhất theo Cosine Similarity.
    """
    t0 = time.time()
    q_vec = embedder.encode_query(query_text)
    encode_ms = (time.time() - t0) * 1000
    
    search_query = """
    CALL db.index.vector.queryNodes('article_embeddings_idx', $top_k, $vector)
    YIELD node, score
    RETURN node.node_id AS node_id,
           node.title AS title,
           node.full_text AS full_text,
           score
    ORDER BY score DESC
    """
    
    t1 = time.time()
    records = client.execute_read(search_query, {"top_k": top_k, "vector": q_vec})
    search_ms = (time.time() - t1) * 1000
    
    results = []
    for rank, r in enumerate(records, 1):
        preview = (r["full_text"] or "").replace("\n", " ")[:150]
        results.append({
            "Rank": rank,
            "Score": round(r["score"], 4),
            "Node ID": r["node_id"],
            "Tiêu đề": r["title"],
            "Trích đoạn nội dung": preview + "..."
        })
        
    print(f"Câu hỏi: '{query_text}'")
    print(f"Thời gian: Encode {encode_ms:.1f}ms | Neo4j Search {search_ms:.1f}ms\n")
    return pd.DataFrame(results)

## Đánh Giá Retrieval Trên Các Tình Huống Pháp Lý Thực Tế

In [ ]:
# Tình huống 1: Lãi suất cho vay theo thỏa thuận trong hợp đồng vay tiền cá nhân
df_1 = vector_search("mức trần lãi suất cho vay theo thỏa thuận là bao nhiêu phần trăm")
display(df_1)

In [ ]:
# Tình huống 2: Bồi thường thiệt hại do lái xe gây tai nạn giao thông (Nguồn nguy hiểm cao độ)
df_2 = vector_search("trách nhiệm bồi thường thiệt hại do phương tiện giao thông vận tải gây ra là nguồn nguy hiểm cao độ")
display(df_2)

In [ ]:
# Tình huống 3: Chơi họ, hụi, biêu, phường và xử lý khi chủ họ bỏ trốn
df_3 = vector_search("nghĩa vụ của chủ họ khi tổ chức họ hụi biêu phường và lãi suất thỏa thuận")
display(df_3)

## Graph Traversal (Mở Rộng Đồ Thị Đa Tầng Qua Quan Hệ REFERENCES)

Khác với RAG thông thường chỉ dừng lại ở đoạn văn bản thô, Legal Knowledge Graph cho phép từ `Article` mở rộng ngay lập tức qua đồ thị:
- Chi tiết các **Khoản** (`Clause`) và **Điểm** (`Point`) trực thuộc qua quan hệ `CONTAINS`.
- Tất cả các Điều luật hoặc văn bản được dẫn chiếu qua quan hệ **`REFERENCES`** (từ cấp Điều, Khoản hoặc Điểm).

In [ ]:
def expand_article_context(article_node_id: str):
    """
    Mở rộng ngữ cảnh của 1 Article: lấy toàn bộ cấu trúc con và các quan hệ REFERENCES (đa tầng).
    """
    query = """
    MATCH (a:Article {node_id: $article_id})
    OPTIONAL MATCH (a)-[:CONTAINS]->(c:Clause)
    OPTIONAL MATCH (c)-[:CONTAINS]->(p:Point)
    OPTIONAL MATCH (a)-[:REFERENCES]->(ref_art)
    OPTIONAL MATCH (c)-[:REFERENCES]->(ref_cls)
    OPTIONAL MATCH (p)-[:REFERENCES]->(ref_pt)
    RETURN a.node_id AS article_id,
           a.title AS article_title,
           count(DISTINCT c) AS total_clauses,
           count(DISTINCT p) AS total_points,
           collect(DISTINCT {from_unit: a.node_id, target: ref_art.node_id}) +
           collect(DISTINCT {from_unit: c.node_id, target: ref_cls.node_id}) +
           collect(DISTINCT {from_unit: p.node_id, target: ref_pt.node_id}) AS all_references
    """
    data = client.execute_read(query, {"article_id": article_node_id})
    if not data:
        return f"Không tìm thấy node: {article_node_id}"
    
    row = data[0]
    clean_refs = [r for r in row['all_references'] if r.get('target') is not None]
    print(f"=== Article: {row['article_id']} — {row['article_title']} ===")
    print(f"• Cấu trúc trực thuộc : {row['total_clauses']} Khoản, {row['total_points']} Điểm")
    print(f"• Các dẫn chiếu REFERENCES ({len(clean_refs)}): {clean_refs}")

# Thử nghiệm mở rộng cho Điều 468 BLDS (Lãi suất)
expand_article_context("BLDS_2015_D468")

# Thử nghiệm mở rộng cho Điều 601 BLDS (Nguồn nguy hiểm cao độ)
expand_article_context("BLDS_2015_D601")